### Importing pyspark libraries/Functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### To check the mount points

In [0]:
display(dbutils.fs.mounts())

## To validate the mount points

## Customer Data cleansing

In [0]:
filePath = '/mnt/bayerhackathon/squad2-bhargav/bronze/customer.csv'
df_customer=spark.read.format('csv').option('header','true').load(filePath)

In [0]:
df_customer.display()

## Filter phone column whose values are blank

In [0]:
df_customer=df_customer.filter(col('phone') != 'none')
df_customer.display()

### Orders data


In [0]:
filePath = '/mnt/bayerhackathon/squad2-bhargav/bronze/order.csv'
df_order=spark.read.format('csv').option('header','true').load(filePath)

In [0]:
df_order.display()

### Filter the rows in orders table whose phone column containing blanks or nulls in customer table

In [0]:
df_order_new=df_customer.join(df_order,df_customer.customer_id==df_order.customer_id,'inner').select(df_order.customer_id,df_order.order_id,df_order.order_date,df_order.order_channel,df_order.store_code,df_order.total_purchase_value,df_order.state,df_order.order_country)

### Filtered data of orders based on customer phone column

In [0]:
df_order_new.display()

Order Line Cleansing

In [0]:
filePath = '/mnt/bayerhackathon/squad2-bhargav/bronze/order_line.csv'
df_order_line=spark.read.format('csv').option('header','true').load(filePath)

In [0]:
df_order_line.display()

### Remove the Rows in order line which are filtered on the above criteria(based on phone col blanks)

In [0]:
df_order_line_new=df_order_new.join(df_order_line,df_order_new.order_id==df_order_new.order_id,'inner').select(df_order_line.order_line_id,df_order_line.order_id,df_order_line.product,df_order_line.quantity,df_order_line.price,df_order_line.order_currency)

In [0]:
df_order_line_new.display()

##Total_purchase_value column update
### order.Total_purchase_value=order.Total_purchase_value+order_line.price

### Casting the string columns to float

In [0]:
df_order_new=df_order_new.withColumn("total_purchase_value",df_order_new.total_purchase_value.cast(FloatType()))

In [0]:
df_order_line_new=df_order_line_new.withColumn("price",df_order_line_new.price.cast(FloatType()))

In [0]:
df_order_line_new.display()

In [0]:
df_order_new.display()

In [0]:
df_order_new1=df_order_new.join(df_order_line_new,df_order_new.order_id==df_order_line_new.order_id,'inner').select(df_order_new.order_id,df_order_new.customer_id,df_order_new.order_date,df_order_new.order_channel,df_order_new.store_code,df_order_new.state,df_order_new.order_country,(df_order_new.total_purchase_value+df_order_line_new.price).alias('total_purchase_value'))

In [0]:
df_order_new1.display()

### Customer Behaviour Cleaning

In [0]:
filePath = '/mnt/bayerhackathon/squad2-bhargav/bronze/customer_behaviour.csv'
df_customer_behaviour=spark.read.format('csv').option('header','true').load(filePath)

In [0]:
df_customer_behaviour.display()

### filter the rows based on customer table phone column which contain blanks or null

In [0]:
df_customer_behaviour=df_customer_behaviour.join(df_customer,df_customer_behaviour.customer_id==df_customer.customer_id,'inner').select(df_customer_behaviour.customer_id,df_customer_behaviour.order_frequency,df_customer_behaviour.average_order_value,df_customer_behaviour.customer_lifetime_value,df_customer_behaviour.website_visits,df_customer_behaviour.seconds_spent_on_website,df_customer_behaviour.page_views,df_customer_behaviour.cart_abandonment_rate)

In [0]:
df_customer_behaviour.display()

### Customer Address Split

In [0]:
df_customer.display()

In [0]:
df_customer=df_customer.withColumn("Addressline1",split(df_customer.Address,' ')[0]).withColumn("Addressline2",split(df_customer.Address,' ')[1])


In [0]:
df_customer.display()

### Listing of Cleaned Data Frames

In [0]:
#df_customer
#df_order_line_new
#df_order_new1
#df_customer_behaviour


### Moving to Silver Layer with the delta format/Delta tables

In [0]:
df_customer_behaviour.display()

In [0]:
df_customer.write.format("delta").mode("overwrite").option("path","/mnt/bayerhackathon/squad2-bhargav/silver/customer").saveAsTable("customer")

In [0]:
%sql
select * from customer

In [0]:
df_customer_behaviour.write.format("delta").mode("overwrite").option("path","/mnt/bayerhackathon/squad2-bhargav/silver/customer_behaviour").saveAsTable("customer_behaviour")

In [0]:
%sql

select * from customer_behaviour

In [0]:
df_order_new1.display()

In [0]:
df_order_new1.write.format("delta").mode("overwrite").option("path","/mnt/bayerhackathon/squad2-bhargav/silver/orders").saveAsTable("orders")

In [0]:
%sql
select * from orders

In [0]:

df_order_line_new.write.format("delta").mode("overwrite").option("path","/mnt/bayerhackathon/squad2-bhargav/silver/order_line").saveAsTable("order_line")

In [0]:
%sql
select * from order_line